In [ ]:
import os
import datetime

import torch
import matplotlib.pyplot as plt
from torch import optim
from torch.nn import functional as F
from torchvision.utils import save_image
import numpy as np
import torch
from torch.utils.data import DataLoader
import h5py

from utils.utils import save_model
from models import VAE_dsprites, VAE_shapes
from datasets import ShapesDataset, DSpritesDataset
import config

In [ ]:
RESULTS_PATH = config.results_path
DATASET = config.dataset_name 
VAE_MODE = config.vae_mode 

if DATASET not in ["3d_shapes", "dsprites"]:
    raise ValueError("DATASET must be one of 'dsprites', '3d_shapes'")

if VAE_MODE not in ["default", "beta"]:
    raise ValueError("VAE_MODE must be one of 'default', 'beta'")

CHAMPION_MODEL_NAME = config.champion_model_name

IMG_WIDTH_HEIGHT = config.img_width_height
NO_CHANNELS = config.no_channels
DISENTANGLEMENT_RANGE = config.disentanglement_range
DISENTANGLEMENT_STEPS = config.disentanglement_steps

print("---SETTINGS---")
print(f"results dir: {RESULTS_PATH}")
print(f"dataset: {DATASET}")
print(f"vae mode: {VAE_MODE}")

In [ ]:
hyperparameters = {
    "device": 'cuda' if torch.cuda.is_available() else 'cpu',
    "train_val_split": config.train_val_split,
    
    "lr": config.lr, 
    "batch_size": config.batch_size, 
    "num_epochs": config.num_epochs,
    
    "log_interval": config.log_interval,
    
    "early_stopping_rounds": config.early_stopping_rounds,
    "early_stopping_min_delta": config.early_stopping_min_delta,
    
    "latent_dim": config.latent_dim, # proposed value by FactorVAE paper
    "beta": config.beta, # proposed value by FactorVAE paper
}

print("\n---HYPERPARAMETERS---")
for key, value in hyperparameters.items():
    print(f"{key}: {value}")

# Loading Data

In [ ]:
# LOADING
if DATASET == "dsprites":
    dataset_zip = np.load('./data/dsprites_64x64.npz')
    images = dataset_zip['imgs']
    n_samples = images.shape[0]

elif DATASET == "3d_shapes":
    shapes_dataset = h5py.File('./data/3d_shapes.h5', 'r')
    
    shapes_labels = shapes_dataset['labels']  # array shape [480000,6], float64
    batch_images = shapes_dataset['images']  # array shape [480000,64,64,3], uint8 in range(256)
    images = np.array(batch_images)
    
    shapes_image_shape = batch_images.shape[1:]  # [64,64,3]
    shapes_label_shape = shapes_labels.shape[1:]  # [6]
    n_samples = shapes_labels.shape[0]

else:
    raise ValueError("DATASET must be one of 'dsprites', '3d_shapes'")


# CREATE TRAIN AND TEST SPLIT
images_shuffled = images[np.random.permutation(n_samples)]

split = int(hyperparameters['train_val_split'] * n_samples)
train_imgs = images_shuffled[:split]
val_imgs  = images_shuffled[split:]


# CREATE DATALOADERS
if DATASET == "dsprites":
    train_loader = DataLoader(DSpritesDataset(train_imgs), batch_size=hyperparameters['batch_size'])
    val_loader = DataLoader(DSpritesDataset(val_imgs), batch_size=hyperparameters['batch_size'])

elif DATASET == "3d_shapes":
    train_loader = DataLoader(ShapesDataset(train_imgs), batch_size=hyperparameters['batch_size'])
    val_loader = DataLoader(ShapesDataset(val_imgs), batch_size=hyperparameters['batch_size'])

else:
    raise ValueError("DATASET must be one of 'dsprites', '3d_shapes'")


# Model & Optimizer

In [ ]:
if DATASET == "dsprites":
    print("using dsprites dataset")
    model = VAE_dsprites(latent_dim=hyperparameters['latent_dim']).to(hyperparameters['device'])
elif DATASET == "3d_shapes":
    print("using 3d_shapes dataset")
    model = VAE_shapes(latent_dim=hyperparameters['latent_dim']).to(hyperparameters['device'])
  
optimizer = optim.Adam(model.parameters(), lr=hyperparameters['lr']) # betas: (0.9, 0.999)

# Training functions

In [ ]:
# Reconstruction + KL divergence losses summed over all elements and batch
def loss_function(recon_x, x, mu, logvar):
    # BCE = F.binary_cross_entropy(recon_x, x.view(-1, IMG_WIDTH_HEIGHT**2), reduction='sum')
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')

    # see Appendix B from VAE paper:
    # Kingma and Welling. Auto-Encoding Variational Bayes. ICLR, 2014
    # https://arxiv.org/abs/1312.6114
    # 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) # -0.5 since we use logvar instead of sigma^2

    return BCE + hyperparameters['beta'] * KLD, BCE, hyperparameters['beta'] * KLD

def train(epoch):
    model.train()
    train_loss_total= train_bce_loss_total = train_kld_loss_total = 0
    for batch_idx, (data) in enumerate(train_loader): # (data, _) if label is present
        data = data.to(hyperparameters['device'])
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        # print(f"Reconstructed batch shape: {recon_batch.shape}, Data shape: {data.shape}")
        loss, bce_loss, kld_loss = loss_function(recon_batch, data, mu, logvar)
        loss.backward()
        train_loss_total += loss.item()
        train_bce_loss_total += bce_loss.item()
        train_kld_loss_total += kld_loss.item()
        optimizer.step()
        
        # Optional logging after certain log_interval
        # if batch_idx % hyperparameters['log_interval'] == 0:
        #     print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
        #         epoch, batch_idx * len(data), len(train_loader.dataset),
        #         100. * batch_idx / len(train_loader),
        #         loss.item() / len(data)))


    print('====> Epoch: {} Average train loss: {:.4f}'.format(
          epoch, train_loss_total / len(train_loader.dataset)))
    
    return train_bce_loss_total / len(train_loader.dataset), train_kld_loss_total / len(train_loader.dataset)


def validate(epoch, result_save_path):
    model.eval()
    val_loss_total = val_bce_loss_total = val_kld_loss_total = 0
    with torch.no_grad():
        for i, (data) in enumerate(val_loader): 
            data = data.to(hyperparameters['device'])
            recon_batch, mu, logvar = model(data)
            loss, bce_loss, kld_loss = loss_function(recon_batch, data, mu, logvar)
            val_loss_total += loss.item()
            val_bce_loss_total += bce_loss.item()
            val_kld_loss_total += kld_loss.item()
            if i == 0:
                n = min(data.size(0), 8)
                comparison = torch.cat([data[:n], recon_batch.view(hyperparameters['batch_size'], NO_CHANNELS, IMG_WIDTH_HEIGHT, IMG_WIDTH_HEIGHT)[:n]])
                save_image(comparison.cpu(), os.path.join(result_save_path, f"reconstruction_{str(epoch)}.png"), nrow=n)

    val_loss_total /= len(val_loader.dataset)
    print('====> Epoch: {} Average validation loss: {:.4f}'.format(epoch, val_loss_total))

    return val_bce_loss_total / len(val_loader.dataset), val_kld_loss_total / len(val_loader.dataset)

# Training

In [ ]:
result_save_path = os.path.join(RESULTS_PATH, str(datetime.datetime.now()))
if not os.path.exists(result_save_path):
    os.makedirs(result_save_path)

train_bce_loss = []
train_kld_loss = []
val_bce_loss = []
val_kld_loss = []
no_epochs_no_improvement = 0
best_val_loss = np.inf

for epoch in range(1, hyperparameters['num_epochs'] + 1):
    train_losses = train(epoch)
    train_bce_loss.append(train_losses[0])
    train_kld_loss.append(train_losses[1])

    val_losses = validate(epoch, result_save_path)
    val_bce_loss.append(val_losses[0])
    val_kld_loss.append(val_losses[1])
    val_loss = val_losses[0] + val_losses[1]

    with torch.no_grad():
        sample = torch.randn(64, hyperparameters['latent_dim']).to(hyperparameters['device'])
        sample = model.decode(sample).cpu()
        save_image(sample.view(64, NO_CHANNELS, IMG_WIDTH_HEIGHT, IMG_WIDTH_HEIGHT), os.path.join(result_save_path, f"sample_{str(epoch)}.png"))
    
    # early stopping:
    if best_val_loss - val_loss > hyperparameters['early_stopping_min_delta']: # only when there is a real improvement
        save_model(model, os.path.join(result_save_path, f"{CHAMPION_MODEL_NAME}.pth"))
        no_epochs_no_improvement = 0
        best_val_loss = val_loss
    else:
        no_epochs_no_improvement += 1
        print(f"No improvement for {no_epochs_no_improvement} epochs.")
        if no_epochs_no_improvement >= hyperparameters['early_stopping_rounds']:
            print(f"No improvement for {no_epochs_no_improvement} epochs. Aborted training due to early stopping.")
            break

## Visualize Loss

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8))
epochs = list(range(len(train_bce_loss)))

# 1: Train BCE
axs[0, 0].plot(epochs, train_bce_loss, label='Train BCE', color='blue')
axs[0, 0].set_title('Train BCE Loss')
axs[0, 0].set_xlabel('Epoch')
axs[0, 0].set_ylabel('Loss')
axs[0, 0].grid(True)

# 2: Train KLD
axs[0, 1].plot(epochs, train_kld_loss, label='Train KLD', color='orange')
axs[0, 1].set_title('Train KLD Loss')
axs[0, 1].set_xlabel('Epoch')
axs[0, 1].set_ylabel('Loss')
axs[0, 1].grid(True)

# 3: Val BCE
axs[1, 0].plot(epochs, val_bce_loss, label='Val BCE', color='green')
axs[1, 0].set_title('Val BCE Loss')
axs[1, 0].set_xlabel('Epoch')
axs[1, 0].set_ylabel('Loss')
axs[1, 0].grid(True)

# 4: Val KLD
axs[1, 1].plot(epochs, val_kld_loss, label='Val KLD', color='red')
axs[1, 1].set_title('Val KLD Loss')
axs[1, 1].set_xlabel('Epoch')
axs[1, 1].set_ylabel('Loss')
axs[1, 1].grid(True)

plt.tight_layout()
plt.show()